# Notebook 01: Static Embeddings

## Why This Matters
Before transformers, the dominant way to represent words as vectors was to learn a static lookup table from co-occurrence statistics. The methods developed between 2013 and 2017 — Word2Vec, GloVe, FastText — established the core ideas that still run through modern embedding models: meaning lives in geometry, similarity is a dot product, and context predicts content.

Word2Vec and GloVe are largely historical today. FastText is still in active production use for multilingual and edge deployments. But all three are worth understanding because they make contrastive learning (SimCLR, CLIP) and sentence embeddings legible — the objectives are the same, just scaled up and applied to different modalities.

## Structure
1. The distributional hypothesis — why co-occurrence encodes meaning
2. Word2Vec — the objective and why it works (narrative)
3. GloVe — global co-occurrence factorization (narrative)
4. The shared failure: one vector per word type
5. FastText — subword embeddings that still ship to production (code)
6. When to use static embeddings today
7. Bridge to contextual embeddings

## What You'll Learn
- Why predicting context from words produces geometrically meaningful vectors
- How GloVe's log-bilinear objective differs from Word2Vec's and why it often wins on analogy tasks
- Why both fail on polysemy — and what "one vector per word type" means in practice
- How FastText's character n-gram decomposition handles OOV and morphologically rich languages
- Where static embeddings still beat contextual models in production


## Background

### The distributional hypothesis

> "You shall know a word by the company it keeps." — J.R. Firth, 1957

Words appearing in similar contexts tend to have similar meanings. *Dog* and *cat* both appear near *pet*, *feed*, and *veterinarian*. *King* and *queen* both appear near *throne*, *reign*, and *crown*. If you capture what contexts a word appears in, you've captured something close to its meaning.

All three static embedding methods operationalize this idea differently — but they share the same foundation.


## 1. Word2Vec — The Objective

Word2Vec (Mikolov et al., 2013) trains a shallow neural network to solve a prediction task. Two variants:

**Skip-gram:** given center word *w*, predict context words within a window.
**CBOW:** given context words, predict the center word.

Skip-gram with negative sampling became the dominant variant. The key objective insight: instead of a full softmax over the vocabulary (intractable at 50k+ words), train a binary classifier — is this a real (center, context) pair from the corpus, or a randomly sampled fake?

```
loss = -log σ(v_center · v_context) - Σ log σ(-v_center · v_negative)
```

You sample k negatives per real pair from a noise distribution `P(w) ∝ freq(w)^0.75` — the 0.75 exponent downweights frequent words like *the* so they don't dominate the negatives.

The result: two embedding matrices (W_in for center words, W_out for context words). W_in is the final word vectors. Each gradient step only touches k+1 rows instead of the full vocabulary.

**Why the vectors are useful:** the model can only win by encoding semantic information in the vector geometry. Words with similar contexts end up nearby because the same context vector must score high against both.

**Famous property:** linear structure emerges — `king - man + woman ≈ queen`. This works because relational pairs (*man/woman*, *king/queen*) differ along consistent dimensions across the embedding space. It's impressive, but only reliable on large corpora and specific relation types.

**Why it's largely historical:** a Word2Vec model trained from scratch is never the right choice today. Pretrained contextual embeddings (BERT, sentence-transformers) outperform it on every downstream task that matters.


## 2. GloVe — Global Co-occurrence Factorization

GloVe (Pennington et al., 2014) takes a different approach: instead of training on one window at a time, build the full co-occurrence matrix X first, where X_ij = how often word j appears in the context of word i across the entire corpus.

The key insight: **ratios of co-occurrence probabilities encode meaning more robustly than raw probabilities.**

```
P(solid | ice) / P(solid | steam) >> 1    # ice relates to solid; steam doesn't
P(gas   | ice) / P(gas   | steam) << 1    # steam relates to gas; ice doesn't
P(water | ice) / P(water | steam) ≈ 1     # both relate to water equally
```

GloVe trains word vectors to predict these log co-occurrence ratios via a weighted least-squares objective:

```
minimize Σ_{i,j} f(X_ij) · (v_i · v_j + b_i + b_j - log X_ij)²
```

Where `f(X_ij)` caps at 1 for very frequent pairs and is 0 for zero co-occurrences — preventing rare and ubiquitous pairs from dominating. Final vectors = W + W_context (both matrices trained, both contribute).

**When GloVe wins over Word2Vec:** analogy tasks and tasks requiring global corpus statistics. Word2Vec's local window training misses long-range co-occurrence patterns that GloVe captures by construction.

**Why it's largely historical:** same reason as Word2Vec. Stanford still hosts pretrained GloVe vectors, and they occasionally appear as baselines in papers, but no new project should start with GloVe.


## 3. The Shared Failure: One Vector Per Word Type

Both Word2Vec and GloVe assign every word a single fixed vector, regardless of context:

```python
embedding["bank"]  # identical vector in:
                   # "I deposited money at the bank"
                   # "we fished along the river bank"
```

This is polysemy — most common words have multiple senses, and a static embedding is forced to average them into a single point. The resulting vector is a blend that doesn't precisely represent any individual sense.

More subtle: even without polysemy, word meaning shifts with context. "Light" in "light reading" vs "light luggage" vs "light a candle" carries different connotations that a single vector cannot encode.

This failure mode is exactly what contextual embeddings fix — and it's the main reason Word2Vec and GloVe are no longer the right tool for most tasks.


## 4. FastText — Subword Embeddings

FastText (Bojanowski et al., 2017) solves a different problem: OOV words and morphology.

Word2Vec and GloVe treat words as atomic units. *Running*, *runs*, *runner*, and *run* are four unrelated entries. A word appearing only once gets a poor embedding. A word never seen in training gets nothing.

FastText represents each word as the **sum (or mean) of its character n-gram embeddings**, with boundary markers:

```
"where" → {"<wh", "whe", "her", "ere", "re>", "<where>"}
v("where") = mean of embeddings for all its n-grams
```

Now *running* and *runner* share embeddings for `<runn`, `runn`, `unni`, etc. An OOV word like *untrainable* can be embedded from its n-grams even if the full word was never in the training corpus.

**Where FastText still ships to production:**
- Multilingual applications — pretrained vectors for 157 languages, including low-resource ones with no BERT model
- Edge / mobile deployment where you can't run a transformer
- Very high-throughput similarity where BERT latency is a dealbreaker
- Morphologically rich languages (Finnish, Turkish, Arabic) where subword structure carries heavy semantic load

The code below uses `fasttext` (the official Facebook Research library) rather than reimplementing from scratch, which is the right call for anything production-adjacent.


In [ ]:
# Self-contained install — run once
# %pip install -q fasttext-wheel numpy scikit-learn matplotlib
# Note: fasttext-wheel is the prebuilt wheel; avoids needing a C++ compiler

In [ ]:
import os, tempfile, collections
import numpy as np
import fasttext
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# fasttext.train_unsupervised expects a plain text file, one sentence per line
corpus_lines = [
    "the king rules the kingdom with wisdom and power",
    "the queen rules the kingdom with grace and wisdom",
    "the prince will become king one day with power",
    "the princess will become queen one day with grace",
    "a dog is a loyal pet that guards the home",
    "a cat is an independent pet that hunts at home",
    "the veterinarian treats sick dogs and cats",
    "a puppy is a young dog that plays and runs",
    "a kitten is a young cat that climbs and plays",
    "the king and queen rule together with wisdom and grace",
    "dogs and cats are common pets in the home",
    "running dogs and climbing cats play in the home every day",
    "the runner trains the young dog to run every day",
    "the kingdom is ruled by a wise and graceful queen",
    "royal dogs and cats live in the kingdom palace",
] * 80  # repeat so fasttext has enough signal on this tiny corpus

corpus_path = tempfile.mktemp(suffix=".txt")
with open(corpus_path, "w") as f:
    f.write("\n".join(corpus_lines))

print(f"Corpus: {len(corpus_lines)} lines written to {corpus_path}")

In [ ]:
# Train a FastText skip-gram model
# dim=50 is the standard for small corpora; production models use 100-300
model = fasttext.train_unsupervised(
    corpus_path,
    model="skipgram",   # or "cbow"
    dim=50,
    epoch=50,
    lr=0.05,
    wordNgrams=1,
    minCount=1,
    minn=3,             # min character n-gram length
    maxn=6,             # max character n-gram length
    verbose=0,
)

print(f"Vocabulary size: {len(model.words)}")
print(f"Embedding dimension: {model.get_dimension()}")
print(f"Sample words: {model.words[:10]}")

In [ ]:
# Word similarity — standard cosine distance
def most_similar(word, model, n=5):
    neighbors = model.get_nearest_neighbors(word, k=n)
    return [(w, round(sim, 3)) for sim, w in neighbors]

print("Most similar words (fasttext cosine similarity):")
print("-" * 50)
for target in ["king", "queen", "dog", "cat", "wisdom"]:
    similar = most_similar(target, model, n=4)
    print(f"  {target:12s}: {similar}")

In [ ]:
# The key FastText feature: OOV word embeddings via n-gram composition
print("OOV word embedding (words never seen in training):")
print("-" * 55)

oov_words = [
    ("kingdoms",    "shares n-grams with 'kingdom'"),
    ("queendom",    "shares n-grams with 'queen'"),
    ("doglike",     "shares n-grams with 'dog'"),
    ("supercat",    "shares n-grams with 'cat'"),
    ("xqzjwp",     "random string — almost no known n-grams"),
]

for word, note in oov_words:
    in_vocab = word in model.words
    vec = model.get_word_vector(word)
    norm = np.linalg.norm(vec)
    print(f"  {word:12s}  in_vocab={str(in_vocab):<5}  vec_norm={norm:.3f}  ({note})")

print()
print("'xqzjwp' has near-zero norm — correct: unknown n-grams → near-zero embedding")
print("All others get meaningful vectors from shared n-grams, even as OOV.")

In [ ]:
# Analogy arithmetic: a - b + c ≈ ?
def analogy(a, b, c, model, n=3):
    va = model.get_word_vector(a)
    vb = model.get_word_vector(b)
    vc = model.get_word_vector(c)
    query = vb - va + vc
    query /= np.linalg.norm(query) + 1e-8

    exclude = {a, b, c}
    sims = []
    for word in model.words:
        if word in exclude:
            continue
        v = model.get_word_vector(word)
        v /= np.linalg.norm(v) + 1e-8
        sims.append((word, float(np.dot(query, v))))
    return sorted(sims, key=lambda x: -x[1])[:n]

print("Analogy queries (a : b :: c : ?)  — small corpus, results are approximate")
print("-" * 60)
for a, b, c in [("man", "king", "woman"), ("dog", "puppy", "cat"), ("king", "wisdom", "queen")]:
    results = analogy(a, b, c, model, n=3)
    print(f"  {a} → {b},  {c} → ?   {[w for w, _ in results]}")

In [ ]:
# Visualize embedding space with PCA
words_to_plot = [w for w in model.words if len(w) > 2][:40]
vecs = np.array([model.get_word_vector(w) for w in words_to_plot])

pca = PCA(n_components=2)
coords = pca.fit_transform(vecs)

groups = {
    "royalty":   {"king", "queen", "prince", "princess", "kingdom"},
    "pets":      {"dog", "cat", "puppy", "kitten", "dogs", "cats"},
    "qualities": {"wisdom", "grace", "power", "loyal"},
}
colors = {"royalty": "royalblue", "pets": "darkorange", "qualities": "seagreen"}
word_to_group = {w: g for g, ws in groups.items() for w in ws}

fig, ax = plt.subplots(figsize=(11, 7))
for word, (x, y) in zip(words_to_plot, coords):
    color = colors.get(word_to_group.get(word), "lightgray")
    ax.scatter(x, y, c=color, s=60, zorder=2)
    ax.annotate(word, (x, y), fontsize=8, ha="center", va="bottom",
                xytext=(0, 4), textcoords="offset points")

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=g) for g, c in colors.items()] +
                  [Patch(color="lightgray", label="other")], loc="upper right")
ax.set_title("FastText Embeddings — PCA Projection")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Loading Pretrained FastText Vectors

For anything real, use the pretrained Common Crawl or Wikipedia vectors rather than training from scratch. Facebook Research distributes 157-language vectors.

```python
import fasttext
import fasttext.util

# Download English 300-dim Common Crawl vectors (~4GB)
fasttext.util.download_model("en", if_exists="ignore")
model = fasttext.load_model("cc.en.300.bin")

# Or reduce dimension for faster inference
fasttext.util.reduce_model(model, 100)

# Same API as a trained model
model.get_nearest_neighbors("quantum")
model.get_word_vector("unseen_technical_term")  # OOV handled automatically
```

Available languages: https://fasttext.cc/docs/en/crawl-vectors.html

For sentence-level similarity (the more common production need today), sentence-transformers are now the right choice — but fastText vectors remain competitive for token-level tasks and low-latency retrieval.


## 6. When to Use Static Embeddings Today

| Use Case | Recommended |
|----------|-------------|
| Token-level similarity at high throughput | FastText pretrained |
| Low-resource languages (157 available) | FastText pretrained |
| Edge / mobile with no transformer runtime | FastText pretrained |
| Morphologically rich languages | FastText pretrained |
| General sentence similarity | sentence-transformers (SBERT) |
| Document retrieval / RAG | sentence-transformers + vector DB |
| Any task with a GPU | Contextual embeddings |

Word2Vec and GloVe pretrained vectors (Google News, Stanford GloVe) are still downloadable and occasionally useful as cheap baselines, but shouldn't be the first choice for any new project.


## 7. Bridge to Contextual Embeddings

Static embeddings share one fundamental limitation regardless of algorithm: **one vector per word type, regardless of context**.

```python
model.get_word_vector("bank")  # identical vector in:
                               # "I went to the bank to deposit money"
                               # "we sat on the river bank at sunset"
```

This matters in practice — sentiment analysis, NER, coreference resolution all depend on sense disambiguation that static embeddings can't provide.

**ELMo** (2018) was the first widely-deployed fix: a bidirectional LSTM over the full sequence, producing a different vector per token per sentence. It improved nearly every NLP benchmark of its era.

**BERT** (2018) generalized this with a transformer encoder and masked language model pretraining, producing deeply contextual representations that became the foundation for nearly all NLP for the following years.

**Sentence-BERT** (2019) adapted BERT for efficient sentence-level similarity — pooling token representations into a fixed-size sentence vector that can be indexed and searched at scale.

Notebook 02 covers the transition: why context dependence matters, how BERT's masked LM objective produces it, and how sentence-transformers make BERT-quality embeddings practical for retrieval.


## Summary

| Method | Year | Key Idea | Status |
|--------|------|----------|--------|
| Word2Vec | 2013 | Predict context from word via negative sampling | Historical — understand the objective, don't deploy |
| GloVe | 2014 | Factorize global log co-occurrence matrix | Historical — occasionally useful as baseline |
| FastText | 2017 | Subword n-gram composition; OOV-capable | **Active** — multilingual, edge, high-throughput |
| Static all | — | One vector per word type | Superseded for most tasks by contextual models |

**Why this notebook still matters:**
- Negative sampling is the same objective used by SimCLR, MoCo, and CLIP (Notebooks 06–09) — just applied to image pairs instead of word pairs
- The OOV problem fastText solves reappears in BPE tokenization — both decompose unknown units into subcomponents
- Analogy arithmetic (`king - man + woman`) is the same geometric reasoning used in concept vector steering of LLMs
- Embedding drift detection (Notebook 16) monitors the same vector space geometry, just over time in production

**Next:** Notebook 02 — Contextual Embeddings. ELMo established the idea; BERT made it standard; sentence-transformers made it practical for retrieval at scale.
